In [2]:
import netket as nk
import jax.numpy as jnp

In [3]:
import jax
import jax.numpy as jnp
import numpy as np

from functools import partial


@partial(jax.jit, static_argnums=(1,))
def smallest_vecs(basis, n):
    dim = basis.shape[-1]
    r = np.arange(-n, (n + 1))  # produces O(n*n*n) numbers

    vecs = jnp.array(np.meshgrid(*(dim * (r,)))).T.reshape(-1, dim)
    return vecs


def ewald_coulomb(x, L, alpha, sdim=3, kmax=20, cutoff=None):
    """Ewald summation for cubic lattices."""

    if sdim != 3:
        raise ValueError("Ewald summation only implemented for 3 spatial dimensions.")

    if cutoff is None:
        cutoff = 2.0 * L

    N = x.shape[0] // sdim
    x = x.reshape(-1, N, sdim)

    dists = x[..., :, None, :] - x[..., None, :, :]
    V = L**sdim

    def real_sum(dists):
        """Real space part of Ewald summation"""

        # vec(R)=0
        mask = 1 - jnp.eye(N)
        d = jnp.linalg.norm(dists, axis=-1)
        res1 = jnp.where(
            mask[jnp.newaxis, :], jax.scipy.special.erfc(alpha * d) / d, 0
        ).reshape(-1)

        # vec(R)=L * vec(n)
        Rn = smallest_vecs(jnp.eye(sdim), 2) * L
        temp = jnp.linalg.norm(Rn, axis=-1)
        Rn = jnp.where(temp[:, None] <= cutoff, Rn, 0)  # exlude too far away neighbours
        temp = jnp.linalg.norm(Rn, axis=-1)
        d = jnp.linalg.norm(dists[..., None, :] + Rn, axis=-1)

        res2 = jnp.where(temp == 0, 0, jax.scipy.special.erfc(alpha * d) / d).reshape(
            -1
        )  # Rn=0 is special

        res = jnp.sum(res1) + jnp.sum(res2)
        return res

    def recip_sum(x):
        """Reciprocal space part of Ewald summation"""

        # vec(k)=1/L * vec(n)
        Gm = smallest_vecs(jnp.eye(sdim), kmax) / L
        # |vec(k)|>kmax
        G = jnp.linalg.norm(Gm, axis=-1, keepdims=True)
        Gm = jnp.where(G <= kmax / L, Gm, 0)
        G = jnp.linalg.norm(Gm, axis=-1)
        G2 = (G**2).reshape(-1)

        t1 = jnp.where(G2 == 0, 0, jnp.exp(-G2 * jnp.pi**2 / (alpha**2)) / G2)
        # structure factor
        S = jnp.sum(
            jnp.where(
                G2 == 0,
                0,
                jnp.exp(1j * 2 * jnp.pi * jnp.einsum("...ij,kj->...ik", x, Gm)),
            ),
            axis=-2,
        ).reshape(-1)

        t2 = jnp.abs(S) ** 2

        res = 1 / (V * jnp.pi) * jnp.sum(t1 * t2)
        return res

    def self_energy():
        return -N * alpha / jnp.sqrt(jnp.pi)

    def constant():
        V = L**sdim
        return -(N**2) * jnp.pi / (2 * alpha**2 * V)

    ereal = real_sum(dists)
    erec = recip_sum(x)
    eself = self_energy()
    const = constant()

    return 0.5 * ereal + 0.5 * erec + eself + const


In [4]:
import flax.linen as nn

import numpy as np
from typing import Any, Callable, Tuple


class MLP(nn.Module):
    out_dim: int
    hidden_layers: Tuple[int]
    activation: Callable = nn.gelu
    output_activation: Callable = None
    last_bias: bool = True
    last_linear: bool = True
    auto_width: bool = False
    dtype: Any = float
    kwargs: dict = None

    @nn.compact
    def __call__(self, x):
        in_dim = x.shape[-1]
        kwargs = self.kwargs if self.kwargs is not None else {}

        if self.auto_width and len(self.hidden_layers) == 1:
            n_hidden = self.hidden_layers[0]
            qs = [k / n_hidden for k in range(n_hidden + 1)]
            dims = [int(np.round(in_dim ** (1 - q) * self.out_dim**q)) for q in qs]
        else:
            dims = [in_dim, *self.hidden_layers, self.out_dim]

        for k in range(len(dims) - 1):
            last = k + 2 == len(dims)
            bias = not last or self.last_bias or not self.last_linear

            x = nn.Dense(
                dims[k + 1],
                use_bias=bias,
                param_dtype=self.dtype,
                name=f"linear{k+1}",
                **kwargs,
            )(x)

            if self.output_activation is not None:
                x = self.output_activation(x)

            if not last or not self.last_linear and not self.output_activation:
                x = self.activation(x)

        return x


In [5]:
import flax.linen as nn
import scipy
import numpy as np
from typing import Any, Callable, Sequence, Tuple
import jax.numpy as jnp


def slogdet(x):
    sign, logdet = jnp.linalg.slogdet(x)
    return sign, logdet


class LogSlaterDet(nn.Module):
    n_per_spin: Tuple[int]
    orbitals: Sequence[Callable[..., Any]]
    return_signs: bool = False
    add_signs: bool = True

    @nn.compact
    def __call__(self, x, orbs=None, mask=True):
        """
        x: (..., N, sdim) Particle positions on which to evaluate the orbtials.
        orbs: (...,N_orbs) Orbital aware Jastrow factor
        mask: whether or not to make the Slater determinant block diagonal
        """

        if not self.orbitals:
            raise ValueError(f"Empty LogSlaterDet module {self.name}.")

        assert (
            x.ndim == 3
        ), f"Got shape {x.shape} where (..., N_particles, sdim) was expected."
        N = x.shape[-2]

        mat = []
        for key, value in self.orbitals.items():
            o = value(x)
            mat.append(o)

        mat = jnp.concatenate(mat, axis=-1)

        assert mat.shape[-2] == mat.shape[-1], (
            f"The matrix constructed from the orbtials has shape {mat.shape} and is "
            f"non-square."
        )

        if mask:
            """mask to make sure that off diagonal blocks are zero"""
            x = jnp.tile(
                jnp.array(self.n_per_spin[0] * [1] + self.n_per_spin[1] * [0])[:, None],
                (1, self.n_per_spin[0]),
            )
            y = jnp.tile(
                jnp.array(self.n_per_spin[0] * [0] + self.n_per_spin[1] * [1])[:, None],
                (1, self.n_per_spin[1]),
            )
            mask = jnp.concatenate((x, y), axis=-1)

            # logdet works in normal space
            mat = mask[..., :, :] * jnp.exp(mat)

        if orbs is not None:
            mat = mat * jnp.exp(orbs)
        assert mat.shape[-1] == mat.shape[-2], (
            f"The matrix constructed from the orbtials has shape {mat.shape} and is "
            f"non-square."
        )

        signs, logslaterdet = jnp.linalg.slogdet(mat)
        # normalization
        log_norm = 0.5 * np.log(scipy.special.factorial(N))
        logslaterdet = logslaterdet - log_norm

        if self.add_signs:
            logslaterdet = logslaterdet + jnp.log(signs)

        if self.return_signs:
            return signs, logslaterdet
        else:
            return logslaterdet


In [6]:
import jax.numpy as jnp
import jax

from flax import linen as nn
from jax.nn.initializers import lecun_normal

from netket.utils import HashableArray
from netket.utils.types import DType, NNInitFunc


def diagonal_init(l, idx, dtype=float):
    def init(key, shape, dtype=dtype):
        eye = jnp.eye(l)
        noise = jax.random.normal(key, eye.shape)
        return eye + noise

    return init


class PW(nn.Module):
    """create plane wave orbitals"""
    kvecs: HashableArray  # rows are k's
    N_orbs: int
    combine: bool = False
    param_init: NNInitFunc = lecun_normal()
    dtype: DType = float

    @nn.compact
    def __call__(self, x):
        """Plane wave orbitals
        x: (..., N, D)
        """

        # ikx #(...,N, N_k)
        res = 1j * jnp.einsum("ij,...j->...i", self.kvecs, x)

        if self.combine:
            params = self.param('PlaneWaveCoefficients', self.param_init, (res.shape[-1], self.N_orbs), self.dtype)
            res = jnp.einsum('ij,...i->...j', params, res)

        return res


In [10]:
import jax.numpy as jnp
import numpy as np

import flax.linen as nn
from jax.nn.initializers import lecun_normal


from typing import Callable

class Networks:
    def __init__(
        self, intermediate_dim, num_intermediate, embedding_dim, activation=nn.gelu
    ):
        """Create networks for the MPNQS
        Args:
            intermediate_dim: dimension of the intermediate coordinates
            num_intermediate: number of hidden layers
            embedding_dim: dimension of the final coordinates of the layer
        """
        self.intermediate_dim = intermediate_dim
        self.num_intermediate = num_intermediate
        self.embedding_dim = embedding_dim
        self.activation = activation

    def Net(self):
        return MLP(
            self.embedding_dim,
            self.num_intermediate * (self.intermediate_dim,),
            activation=self.activation,
        )


class GraphUpdate(nn.Module):
    """Upplies the Particle Attention Graph Updates"""

    networks: Callable
    L: float
    n_up: int
    n_down: int
    attention_dim: int

    @nn.compact
    def __call__(self, x, xij, x0, xij0, i, final):
        """
        x: Current nodes of the graph
        xij: Current edges of the graph
        x0: Initial nodes of the graph
        xij0: Initial edges of the graph
        i: iteration
        final: bool whether it is the last iteration
        """

        # Networks
        phi = self.networks.Net() # edge: message content
        f = self.networks.Net() # node + aggregated messages: new node information
        f2 = self.networks.Net() # attention correlation: attention weight

        M, N, _ = x.shape

        wquery = self.param(
            f"query_{i}",
            lecun_normal(),
            (xij.shape[-1], self.attention_dim),
            np.float64,
        )
        wkey = self.param(
            f"key_{i}", lecun_normal(), (xij.shape[-1], self.attention_dim), np.float64
        )
        # shape of WQ, WK: Real (d_edge x d_attention)

        xquery = jnp.dot(xij, wquery) # q_ij = x_ij W_Q
        xkey = jnp.dot(xij, wkey) # k_ij = x_ij W_k
        # shape: (M,N,N,d_attention)

        weight = f2(jnp.einsum("...ijk,...jlk->...ilk", xquery, xkey) / jnp.sqrt(32))
        # A_il = \sum_j,k q_ijk k_jlk

        message = phi(xij) * weight  # ~ edge info x learned particle attention 

        xnew = jnp.concatenate(
            (x0, f(jnp.concatenate((x, jnp.sum(message, axis=-2)), axis=-1))), axis=-1
        ) # x_i^(n+1) = [ x_i^(0), f(x_i^(n), m_i^(n) ]
        if not final:
            g = self.networks.Net()
            xij = jnp.concatenate(
                (xij0, g(jnp.concatenate((xij, message), axis=-1))), axis=-1
            ) # x_ij^(n+1) = [ x_ij^(0), g(x_ij^(n), m_ij^(n) ]
        return xnew, xij


class MPNN(nn.Module):
    n_up: int
    n_down: int
    embedding_dim: int
    intermediate_dim: int
    num_intermediate: int
    attention_dim: int
    L: float = 1.0
    n_interactions: int = 1

    @nn.compact
    def __call__(self, x):
        assert len(x.shape) == 3
        M, N, sdim = x.shape

        # network generator
        networks = Networks
        network_factory = networks(
            self.intermediate_dim, self.num_intermediate, self.embedding_dim
        )

        # embeddings
        X = nn.Embed(
            num_embeddings=3,
            features=self.embedding_dim,
            dtype=np.float64,
            param_dtype=np.float64,
            embedding_init=lecun_normal(),
        )
        xembed = jnp.tile(X(jnp.array([0, 1, 2]))[None, ...], (M, 1, 1))

        # spin products (assuming first coordinates are spin up and then spin down)
        s = jnp.hstack((self.n_up * [1], self.n_down * [-1]))
        ss = jnp.outer(s, s)[..., None]

        # compute periodic distances and their norm
        dist = x[..., :, None, :] - x[..., None, :, :]
        d = jnp.linalg.norm(
            jnp.sin(jnp.pi / self.L * dist) + jnp.eye(N)[..., None],
            axis=-1,
            keepdims=True,
        ) * (1.0 - jnp.eye(N)[..., None])
        distp = jnp.concatenate(
            (jnp.sin(2 * jnp.pi / self.L * dist), jnp.cos(2 * jnp.pi / self.L * dist)),
            axis=-1,
        )

        # nodes (translation invariance requires no single particle coordinates)
        x0 = jnp.tile(xembed[:, 0, :][:, None, :], (1, N, 1))
        x = jnp.concatenate(
            (x0, jnp.tile(xembed[:, 1, :][:, None, :], (1, N, 1))), axis=-1
        )

        # edges
        xij0 = jnp.concatenate(
            (distp, d, jnp.tile(ss[None, :, :, :], (x.shape[0], 1, 1, 1))), axis=-1
        )
        xij = jnp.concatenate(
            (xij0, jnp.tile(xembed[:, 2, :][:, None, None, :], (1, N, N, 1))), axis=-1
        )

        for i in range(self.n_interactions):
            layer = GraphUpdate(
                network_factory, self.L, self.n_up, self.n_down, self.attention_dim
            )

            x, xij = layer(x, xij, x0, xij0, i, i == self.n_interactions - 1)

        return x


In [8]:
import numpy as np
from jax import numpy as jnp

from flax import linen as nn
from jax.nn.initializers import ones, lecun_normal, variance_scaling


from netket.utils.types import DType, NNInitFunc
from typing import Tuple

def smallest_kvecs(basis, m, n):
    dim = basis.shape[-1]
    r = np.arange(-n, (n + 1))  # produces O(n*n*n) numbers

    vecs = np.array(np.meshgrid(*(dim * (r,)))).T.reshape(-1, dim)
    vecs = vecs[np.argsort(np.linalg.norm(vecs, axis=1))]
    return vecs[:m, :]


class MPNQS(nn.Module):
    n_per_spin: Tuple[int]
    sdim: int = 1
    L: float = 1.0
    paramsre_init: NNInitFunc = lecun_normal()
    params_init: NNInitFunc = variance_scaling(
        0.0001, "fan_in", "truncated_normal"
    )  # lecun_normal()
    scale_init: NNInitFunc = ones
    dtype: DType = jnp.float64

    def setup(self):
        """Construction of plane wave orbitals"""
        k_vec = (
            2
            * np.pi
            / self.L
            * jnp.concatenate(
                (
                    smallest_kvecs(jnp.eye(self.sdim), self.n_per_spin[0], 10),
                    smallest_kvecs(jnp.eye(self.sdim), self.n_per_spin[1], 10),
                ),
                axis=0,
            )
        )
        self.k_vec = k_vec
        pw_basis = {}

        pw_basis["orbtials_up"] = PW(k_vec[: self.n_per_spin[0]], N_orbs=self.n_per_spin[0], combine=True)
        pw_basis["orbtials_down"] = PW(k_vec[self.n_per_spin[0] :], N_orbs=self.n_per_spin[1], combine=True)
        self.orb_basis = pw_basis

        self.MPNN = MPNN(
            *self.n_per_spin,
            embedding_dim=32,
            intermediate_dim=32,
            num_intermediate=2,
            attention_dim=32,
            L=self.L,
            n_interactions=1
        )
        self.slater = LogSlaterDet(n_per_spin=self.n_per_spin, orbitals=self.orb_basis)
        self.net = MLP(out_dim=1, hidden_layers=(32,))

    @nn.compact
    def __call__(self, x):
        """input: x = (..., N*d)"""

        sha = x.shape
        N = sum(self.n_per_spin)
        x = x.reshape(-1, sum(self.n_per_spin), self.sdim)
        M = x.shape[0]

        # backflow
        y = self.MPNN(x)
        prealup = self.param(
            "realup", self.paramsre_init, (y.shape[-1], self.sdim), self.dtype
        )
        pimagup = self.param(
            "imagup", self.params_init, (y.shape[-1], self.sdim), self.dtype
        )
        bf = x + jnp.einsum("kd, ...ik->...id", prealup + 1j * pimagup, y)

        k_vec = jnp.tile(self.k_vec[None, None, :, :], (M, N, 1, 1))

        jastrow = jnp.tile(y[:, :, None, :], (1, 1, N, 1))
        jastrow = jnp.concatenate((jastrow, k_vec), axis=-1)
        jastrow = self.net(jastrow).squeeze(axis=-1)
        psi = self.slater(bf, jastrow)

        return psi.reshape(sha[0])


In [11]:

def mycb(step, logged_data, driver):
    logged_data["acceptance"] = float(driver.state.sampler_state.acceptance)
    return True


N = 14
# unpolarized
n_per_spin = (N // 2, N // 2)
sdim = 3
L = (4 * jnp.pi / 3 * N) ** (1 / 3)
rs = 2.0

hilb = nk.hilbert.Particle(N=N, L=(L, L, L), pbc=True)
sa = nk.sampler.MetropolisGaussian(hilb, sigma=0.04, n_chains=12, n_sweeps=4)

potential = lambda x: ewald_coulomb(x, L, alpha=1.0, kmax=20)
epot = 1 / rs * nk.operator.PotentialEnergy(hilb, potential)
ekin = nk.operator.KineticEnergy(hilb, mass=rs**2)
ha = ekin + epot

model = MPNQS(n_per_spin=n_per_spin, sdim=3, L=L)
vs = nk.vqs.MCState(sa, model, n_samples=8, n_discard_per_chain=2)
# vs.chunk_size = 1

op = nk.optimizer.Sgd(0.05)
sr = nk.optimizer.SR(diag_shift=0.0001)

log = nk.logging.JsonLog("HEG_14particles_test", save_params_every=7)

gs = nk.VMC(ha, op, variational_state=vs, preconditioner=sr)
gs.run(n_iter=2, callback=mycb, out=log)


/Users/azhar/venvs/mp-nqs-original/lib/python3.10/site-packages/netket/vqs/mc/mc_state/state.py:59: UserWarning: n_samples=8 (8 per MPI rank) does not divide n_chains=12, increased to 12 (12 per MPI rank)
  warnings.warn(
  0%|          | 0/2 [00:00<?, ?it/s]/Users/azhar/venvs/mp-nqs-original/lib/python3.10/site-packages/netket/jax/utils.py:96: FutureWarning: jax.tree_leaves is deprecated, and will be removed in a future release. Use jax.tree_util.tree_leaves instead.
  return any(jax.tree_leaves(jax.tree_map(is_complex, pars)))
/Users/azhar/venvs/mp-nqs-original/lib/python3.10/site-packages/netket/jax/utils.py:96: FutureWarning: jax.tree_leaves is deprecated, and will be removed in a future release. Use jax.tree_util.tree_leaves instead.
  return any(jax.tree_leaves(jax.tree_map(is_complex, pars)))
/Users/azhar/venvs/mp-nqs-original/lib/python3.10/site-packages/netket/jax/utils.py:96: FutureWarning: jax.tree_leaves is deprecated, and will be removed in a future release. Use jax.tree_u

(JsonLog('HEG_14particles_test', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    0.0014848709106445312
   	Params: 0.001130819320678711,)